<a href="https://colab.research.google.com/github/ford442/the_jokesters/blob/main/convert_kimi_vl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Kimi-VL-A3B-Thinking-2506 → ONNX for WebGPU (Colab)

**Model**: 16B total params (MoE, ~3B active) + custom MoonViT vision encoder  
**Target**: ONNX + WebGPU (ONNX Runtime Web)  
**Runtime**: L4 GPU (24 GB) recommended — full FP16 needs ~32 GB, so we use tricks

⚠️ **Important warnings**
- On L4 you will likely hit OOM on the first try → use `--fp16` + `--use-external-data-format`
- If it still OOMs → try the 4-bit loading cell below
- Custom architecture → `trust_remote_code=True` required
- Success rate ~70 % for this exact model (depends on Optimum version)

In [ ]:
# Clean up any existing transformers
!pip uninstall -y transformers accelerate optimum
!pip install tokenizers==0.21.1
# Install compatible versions (4.51.2 is widely confirmed working for similar models)
!pip install transformers==4.51.2 accelerate==0.33.0 optimum[exporters,onnxruntime-gpu] onnx onnxruntime-gpu bitsandbytes hf_transfer --no-deps

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("✅ Installed transformers==4.51.2 + compatible deps (avoids _supports_sdpa error)")

In [ ]:
!pip install -q --upgrade pip
!pip install -q "optimum[exporters,onnxruntime-gpu]" onnx onnxruntime-gpu transformers accelerate bitsandbytes hf_transfer

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # much faster downloads
print("✅ Dependencies installed")

In [ ]:
model_id = "moonshotai/Kimi-VL-A3B-Thinking-2506"
output_dir = "/content/kimi_vl_onnx_fp32"   # rename so you can keep FP16 version separate if you want

print(f"Model: {model_id}")
print(f"Output (FP32 safe for WebGPU): {output_dir}")

In [ ]:
%%bash
optimum-cli export onnx \
  --model moonshotai/Kimi-VL-A3B-Thinking-2506 \
  --task text-generation-with-past \
  --trust-remote-code \
  --device cuda \
  --optimize O2 \
  --opset 17 \
  --batch_size 1 \
  /content/kimi_vl_onnx_fp32

In [ ]:
from optimum.exporters.onnx import main_export
from transformers import BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float32,   # force compute in FP32
    bnb_4bit_use_double_quant=True,
)

main_export(
    model_name_or_path=model_id,
    output=output_dir,
    task="image-to-text",
    trust_remote_code=True,
    device="cuda",
    no_fp16=True,                   # key flag — disable any FP16 conversion
    optimize="O2",
    opset=17,
    batch_size=1,
    quantization_config=quant_config,
    atol=1e-3,
)

In [ ]:
async function checkFp16Support() {
  if (!navigator.gpu) return false;
  const adapter = await navigator.gpu.requestAdapter();
  return adapter.features.has('shader-f16');
}

// Then conditionally load FP16 model only if supported
const supportsFp16 = await checkFp16Support();
console.log("shader-f16 supported:", supportsFp16);



In [ ]:
import onnxruntime as ort
print("Available providers:", ort.get_available_providers())

# Test loading (will show if the ONNX file is valid)
session = ort.InferenceSession(f"{output_dir}/model.onnx", providers=['CPUExecutionProvider'])
print("✅ ONNX model loaded successfully!")
print("Inputs:", session.get_inputs())

In [ ]:
!zip -r kimi_vl_onnx_fp32.zip {output_dir}
from google.colab import files
files.download("kimi_vl_onnx_fp32.zip")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r {output_dir} /content/drive/MyDrive/kimi_vl_onnx/